In [1]:
from datetime import datetime, timezone
import json
import time

import pandas as pd
import requests

from project_paths import project_root as get_project_root
from pipeline_config import (
    COUNTRY_CODES,
    EXPECTED_COUNTRY_NAMES,
    GEOCODING_QUERY_OVERRIDES,
    OPEN_METEO_GEOCODING_ENDPOINT as GEOCODING_ENDPOINT,
    normalize_for_match,
)

pd.set_option('display.max_columns', 80)
pd.set_option('display.max_rows', 30)

base_path = get_project_root()
bronze_path = base_path / 'data' / 'bronze'
silver_path = base_path / 'data' / 'silver'
cache_path = base_path / 'data' / 'cache'
reference_path = base_path / 'data' / 'reference'
outputs_tables_path = base_path / 'outputs' / 'tables'

venues_output_path = bronze_path / 'venues_unique.parquet'
geocoding_cache_path = cache_path / 'geocoding_cache.json'
geocode_reference_path = reference_path / 'location_geocode_overrides.csv'
geocodes_output_path = silver_path / 'venue_geocodes.parquet'
geocoding_failures_output_path = outputs_tables_path / 'geocoding_failures.csv'

for path in [silver_path, cache_path, outputs_tables_path]:
    path.mkdir(parents=True, exist_ok=True)

venues_unique = pd.read_parquet(venues_output_path)

{
    'venues_input': 'data/bronze/venues_unique.parquet',
    'geocodes_output': 'data/silver/venue_geocodes.parquet',
    'geocoding_cache_output': 'data/cache/geocoding_cache.json',
    'geocode_reference_input': 'data/reference/location_geocode_overrides.csv',
    'geocoding_failures_output': 'outputs/tables/geocoding_failures.csv',
    'endpoint': GEOCODING_ENDPOINT,
}


{'venues_input': 'data/bronze/venues_unique.parquet',
 'geocodes_output': 'data/silver/venue_geocodes.parquet',
 'geocoding_cache_output': 'data/cache/geocoding_cache.json',
 'geocode_reference_input': 'data/reference/location_geocode_overrides.csv',
 'geocoding_failures_output': 'outputs/tables/geocoding_failures.csv',
 'endpoint': 'https://geocoding-api.open-meteo.com/v1/search'}

# 03b Geocoding vorbereiten

Fuer Wetterabfragen reicht eine Koordinate pro Stadt/Land-Kombination. Deshalb wird der API-Cache auf `location_key` gefuehrt und danach wieder auf die Venue-Zeilen gejoint. Die Abfrage nutzt die Open-Meteo Geocoding API und speichert die Rohentscheidung im JSON-Cache.

In [2]:
def load_geocoding_cache(path):
    if not path.exists():
        return {}
    with path.open('r', encoding='utf-8') as file:
        return json.load(file)


def save_geocoding_cache(cache, path):
    with path.open('w', encoding='utf-8') as file:
        json.dump(cache, file, ensure_ascii=False, indent=2, sort_keys=True)
        file.write('\n')


def load_geocode_reference(path):
    if not path.exists():
        return {}
    reference = pd.read_csv(path)
    reference['latitude'] = pd.to_numeric(reference['latitude'], errors='coerce')
    reference['longitude'] = pd.to_numeric(reference['longitude'], errors='coerce')
    required_columns = {'location_key', 'latitude', 'longitude'}
    missing_columns = required_columns - set(reference.columns)
    if missing_columns:
        raise ValueError(f'Missing geocode reference columns: {sorted(missing_columns)}')
    invalid_rows = reference[reference[['latitude', 'longitude']].isna().any(axis=1)]
    if not invalid_rows.empty:
        invalid_keys = invalid_rows['location_key'].tolist()
        raise ValueError(f'Invalid reference coordinates for location keys: {invalid_keys}')
    return reference.set_index('location_key').to_dict('index')


def choose_best_geocoding_result(results, city_name, country_name):
    if not results:
        return None, 'no_results'

    expected_country_code = COUNTRY_CODES.get(country_name)
    expected_country_names = EXPECTED_COUNTRY_NAMES.get(country_name, {country_name})
    normalized_city = normalize_for_match(city_name)
    normalized_expected_countries = {normalize_for_match(country) for country in expected_country_names}

    scored_results = []
    for position, result in enumerate(results):
        result_country_code = result.get('country_code')
        result_country_name = result.get('country')
        result_name = result.get('name')
        score = 0
        if expected_country_code and result_country_code == expected_country_code:
            score += 100
        if normalize_for_match(result_country_name) in normalized_expected_countries:
            score += 50
        if normalize_for_match(result_name) == normalized_city:
            score += 25
        score -= position
        scored_results.append((score, position, result))

    scored_results.sort(key=lambda item: item[0], reverse=True)
    best_score, _, best_result = scored_results[0]
    quality_flag = 'ok' if best_score >= 120 else 'review'
    return best_result, quality_flag


def build_reference_geocode_entry(row, reference_geocodes, reason):
    location_key = row['location_key']
    reference_entry = reference_geocodes.get(location_key)
    if reference_entry is None:
        return None

    return {
        'location_key': location_key,
        'city_name': row['city_name'],
        'country_name': row['country_name'],
        'query_name': GEOCODING_QUERY_OVERRIDES.get((row['city_name'], row['country_name']), row['city_name']),
        'country_code': COUNTRY_CODES.get(row['country_name']),
        'source': 'reference-geocode',
        'requested_at_utc': datetime.now(timezone.utc).isoformat(),
        'cache_hit': False,
        'fallback_used': True,
        'fallback_reason': reason,
        'geocode_status': 'found',
        'quality_flag': 'ok',
        'latitude': reference_entry['latitude'],
        'longitude': reference_entry['longitude'],
        'matched_name': reference_entry.get('matched_name'),
        'matched_country': reference_entry.get('matched_country'),
        'matched_country_code': reference_entry.get('matched_country_code'),
        'matched_admin1': reference_entry.get('matched_admin1'),
        'matched_timezone': reference_entry.get('matched_timezone'),
        'result_count': 1,
        'failure_reason': None,
        'source_note': reference_entry.get('source_note'),
    }


def cacheable_entry(entry):
    return {key: value for key, value in entry.items() if key != 'cache_hit'}


def geocode_location(row, cache, reference_geocodes, request_delay_seconds=0.3, max_attempts=3):
    location_key = row['location_key']
    city_name = row['city_name']
    country_name = row['country_name']
    query_name = GEOCODING_QUERY_OVERRIDES.get((city_name, country_name), city_name)
    cached_entry = cache.get(location_key)
    if (
        cached_entry is not None
        and cached_entry.get('query_name', city_name) == query_name
        and cached_entry.get('geocode_status') == 'found'
    ):
        entry = dict(cached_entry)
        entry['cache_hit'] = True
        entry['fallback_used'] = bool(entry.get('fallback_used', False))
        return entry

    country_code = COUNTRY_CODES.get(country_name)
    params = {
        'name': query_name,
        'count': 10,
        'language': 'en',
        'format': 'json',
    }
    if country_code:
        params['countryCode'] = country_code

    entry = {
        'location_key': location_key,
        'city_name': city_name,
        'country_name': country_name,
        'query_name': query_name,
        'country_code': country_code,
        'source': 'open-meteo-geocoding',
        'requested_at_utc': datetime.now(timezone.utc).isoformat(),
        'cache_hit': False,
        'fallback_used': False,
    }

    last_error = None
    for attempt in range(1, max_attempts + 1):
        try:
            response = requests.get(GEOCODING_ENDPOINT, params=params, timeout=20)
            entry['request_url'] = response.url
            entry['http_status'] = response.status_code
            response.raise_for_status()
            payload = response.json()
            results = payload.get('results', [])
            result, quality_flag = choose_best_geocoding_result(results, city_name, country_name)
            entry['result_count'] = len(results)
            entry['quality_flag'] = quality_flag

            if result is None:
                entry.update({
                    'geocode_status': 'not_found',
                    'latitude': None,
                    'longitude': None,
                    'matched_name': None,
                    'matched_country': None,
                    'matched_country_code': None,
                    'matched_admin1': None,
                    'matched_timezone': None,
                    'failure_reason': 'Open-Meteo returned no results for this city/country.',
                })
            else:
                entry.update({
                    'geocode_status': 'found',
                    'latitude': result.get('latitude'),
                    'longitude': result.get('longitude'),
                    'matched_name': result.get('name'),
                    'matched_country': result.get('country'),
                    'matched_country_code': result.get('country_code'),
                    'matched_admin1': result.get('admin1'),
                    'matched_timezone': result.get('timezone'),
                    'failure_reason': None if quality_flag == 'ok' else 'Best API result should be reviewed before weather extraction.',
                })
            break
        except requests.RequestException as error:
            last_error = error
            if attempt < max_attempts:
                time.sleep(request_delay_seconds * attempt)
        except ValueError as error:
            last_error = error
            break

    if last_error is not None and entry.get('geocode_status') is None:
        entry.update({
            'geocode_status': 'error',
            'latitude': None,
            'longitude': None,
            'matched_name': None,
            'matched_country': None,
            'matched_country_code': None,
            'matched_admin1': None,
            'matched_timezone': None,
            'result_count': 0,
            'quality_flag': 'error',
            'failure_reason': str(last_error),
        })

    if entry.get('geocode_status') != 'found' or pd.isna(entry.get('latitude')) or pd.isna(entry.get('longitude')):
        fallback = build_reference_geocode_entry(row, reference_geocodes, entry.get('failure_reason'))
        if fallback is not None:
            entry = fallback

    if entry.get('geocode_status') == 'found':
        cache[location_key] = cacheable_entry(entry)

    time.sleep(request_delay_seconds)
    return entry


geocoding_cache = load_geocoding_cache(geocoding_cache_path)
reference_geocodes = load_geocode_reference(geocode_reference_path)
location_requests = (
    venues_unique[['location_key', 'city_name', 'country_name']]
    .drop_duplicates()
    .sort_values(['country_name', 'city_name'])
    .reset_index(drop=True)
)

{
    'unique_locations': len(location_requests),
    'cached_locations_before_run': len(geocoding_cache),
    'reference_locations': len(reference_geocodes),
    'endpoint': GEOCODING_ENDPOINT,
}


{'unique_locations': 53,
 'cached_locations_before_run': 53,
 'reference_locations': 53,
 'endpoint': 'https://geocoding-api.open-meteo.com/v1/search'}

## Open-Meteo API abfragen und Cache aktualisieren

Die API wird nur fuer Location-Keys abgefragt, die noch nicht im Cache liegen. Bei wiederholter Ausfuehrung kommen die Ergebnisse direkt aus `data/cache/geocoding_cache.json`.

In [3]:
geocode_entries = [
    geocode_location(row, geocoding_cache, reference_geocodes)
    for _, row in location_requests.iterrows()
]
save_geocoding_cache(geocoding_cache, geocoding_cache_path)

location_geocodes = pd.DataFrame(geocode_entries)
location_geocodes['latitude'] = pd.to_numeric(location_geocodes['latitude'], errors='coerce')
location_geocodes['longitude'] = pd.to_numeric(location_geocodes['longitude'], errors='coerce')

{
    'locations_requested': len(location_requests),
    'locations_geocoded': int((location_geocodes['geocode_status'] == 'found').sum()),
    'cache_hits_this_run': int(location_geocodes['cache_hit'].sum()),
    'fallbacks_used_this_run': int(location_geocodes['fallback_used'].sum()),
    'cache_entries_after_run': len(geocoding_cache),
    'status_counts': location_geocodes['geocode_status'].value_counts(dropna=False).to_dict(),
    'quality_counts': location_geocodes['quality_flag'].value_counts(dropna=False).to_dict(),
}


{'locations_requested': 53,
 'locations_geocoded': 53,
 'cache_hits_this_run': 39,
 'fallbacks_used_this_run': 0,
 'cache_entries_after_run': 53,
 'status_counts': {'found': 53},
 'quality_counts': {'ok': 53}}

## Venue-Geocodes speichern

Die city-level Geocodes werden zur Venue-Tabelle zurueckgejoint. So bleibt jede Stadionzeile erhalten, nutzt aber denselben Location-Cache wie andere Stadien in derselben Stadt.

In [4]:
geocode_columns = [
    'location_key',
    'latitude',
    'longitude',
    'geocode_status',
    'quality_flag',
    'source',
    'cache_hit',
    'fallback_used',
    'fallback_reason',
    'matched_name',
    'matched_country',
    'matched_country_code',
    'matched_admin1',
    'matched_timezone',
    'result_count',
    'request_url',
    'failure_reason',
    'source_note',
]

for column in geocode_columns:
    if column not in location_geocodes.columns:
        location_geocodes[column] = pd.NA

venue_geocodes = venues_unique.merge(
    location_geocodes[geocode_columns],
    on='location_key',
    how='left',
)
venue_geocodes.to_parquet(geocodes_output_path, index=False)

{
    'venue_rows': len(venue_geocodes),
    'venues_with_coordinates': int(venue_geocodes[['latitude', 'longitude']].notna().all(axis=1).sum()),
    'unique_geocoded_locations': int(location_geocodes.loc[location_geocodes['geocode_status'].eq('found'), 'location_key'].nunique()),
    'venues_using_reference_fallback': int(venue_geocodes['fallback_used'].fillna(False).sum()),
    'geocodes_output': 'data/silver/venue_geocodes.parquet',
}


{'venue_rows': 60,
 'venues_with_coordinates': 60,
 'unique_geocoded_locations': 53,
 'venues_using_reference_fallback': 0,
 'geocodes_output': 'data/silver/venue_geocodes.parquet'}

## Fehlende und unsichere Geocodes dokumentieren

Die Failure-Tabelle enthaelt echte Fehler sowie Ergebnisse mit `quality_flag = review`. Damit sind fehlende oder unsichere Lookups explizit sichtbar, auch wenn die Pipeline weiterlaufen kann.

In [5]:
failure_mask = (
    location_geocodes['geocode_status'].ne('found')
    | location_geocodes['latitude'].isna()
    | location_geocodes['longitude'].isna()
    | location_geocodes['quality_flag'].ne('ok')
)

failure_columns = [
    'location_key',
    'city_name',
    'country_name',
    'country_code',
    'geocode_status',
    'quality_flag',
    'latitude',
    'longitude',
    'matched_name',
    'matched_country',
    'matched_country_code',
    'matched_admin1',
    'result_count',
    'fallback_used',
    'fallback_reason',
    'failure_reason',
    'request_url',
]
for column in failure_columns:
    if column not in location_geocodes.columns:
        location_geocodes[column] = pd.NA
geocoding_failures = location_geocodes.loc[failure_mask, failure_columns].copy()
geocoding_failures.to_csv(geocoding_failures_output_path, index=False)

{
    'geocoding_failures_or_reviews': len(geocoding_failures),
    'failure_status_counts': geocoding_failures['geocode_status'].value_counts(dropna=False).to_dict() if not geocoding_failures.empty else {},
    'failure_quality_counts': geocoding_failures['quality_flag'].value_counts(dropna=False).to_dict() if not geocoding_failures.empty else {},
    'geocoding_failures_output': 'outputs/tables/geocoding_failures.csv',
}


{'geocoding_failures_or_reviews': 0,
 'failure_status_counts': {},
 'failure_quality_counts': {},
 'geocoding_failures_output': 'outputs/tables/geocoding_failures.csv'}

## Qualitaetschecks BD-10

Die Checks bilden die BD-10-Akzeptanzkriterien ab: Die meisten Venues haben Koordinaten, der Cache deckt alle Location-Keys ab und fehlende oder unsichere Geocodes werden als CSV dokumentiert.

In [6]:
coordinate_coverage = venue_geocodes[['latitude', 'longitude']].notna().all(axis=1).mean()
missing_cache_keys = sorted(set(location_requests['location_key']) - set(geocoding_cache.keys()))
failed_locations = location_geocodes[location_geocodes['geocode_status'].ne('found')]

quality_checks_bd10 = {
    'geocodes_output_exists': geocodes_output_path.exists(),
    'geocoding_cache_exists': geocoding_cache_path.exists(),
    'geocoding_failures_output_exists': geocoding_failures_output_path.exists(),
    'venue_rows': len(venue_geocodes),
    'location_rows': len(location_requests),
    'coordinate_coverage': round(float(coordinate_coverage), 4),
    'reference_fallback_locations': sorted(location_geocodes.loc[location_geocodes['fallback_used'].fillna(False), 'location_key'].tolist()),
    'missing_cache_keys': missing_cache_keys,
    'failed_locations': failed_locations[['location_key', 'failure_reason']].to_dict('records'),
}

assert quality_checks_bd10['geocodes_output_exists']
assert quality_checks_bd10['geocoding_cache_exists']
assert quality_checks_bd10['geocoding_failures_output_exists']
assert quality_checks_bd10['venue_rows'] == len(venues_unique)
assert coordinate_coverage >= 0.8
assert not missing_cache_keys
assert len(failed_locations) == 0

quality_checks_bd10


{'geocodes_output_exists': True,
 'geocoding_cache_exists': True,
 'geocoding_failures_output_exists': True,
 'venue_rows': 60,
 'location_rows': 53,
 'coordinate_coverage': 1.0,
 'reference_fallback_locations': [],
 'missing_cache_keys': [],
 'failed_locations': []}

In [7]:
venue_geocodes.head(20)

,venue_key,location_key,stadium_id,stadium_name,stadium_name_clean,city_name,country_name,geocoding_query,match_count,first_match_date,last_match_date,tournaments,latitude,longitude,geocode_status,quality_flag,source,cache_hit,fallback_used,fallback_reason,matched_name,matched_country,matched_country_code,matched_admin1,matched_timezone,result_count,request_url,failure_reason,source_note
0,baku__azerbaijan__baki_olimpiya_stadionu,baku__azerbaijan,4549,Bakı Olimpiya Stadionu,Bakı Olimpiya Stadionu,Baku,Azerbaijan,"Bakı Olimpiya Stadionu, Baku, Azerbaijan",4,2021-06-12,2021-07-03,Euro 2020,40.37767,49.89201,found,ok,open-meteo-geocoding,True,False,<NA>,Baku,Azerbaijan,AZ,Baku City,Asia/Baku,1,https://geocoding-api.open-meteo.com/v1/search...,None,<NA>
1,abidjan__cote_d_ivoire__stade_felix_houphouet_...,abidjan__cote_d_ivoire,117916,Stade Félix Houphouët-Boigny,Stade Félix Houphouët-Boigny,Abidjan,Côte d'Ivoire,"Stade Félix Houphouët-Boigny, Abidjan, Côte d'...",10,2024-01-14,2024-02-10,AFCON 2023,5.35444,-4.00167,found,ok,open-meteo-geocoding,True,False,<NA>,Abidjan,Ivory Coast,CI,Abidjan Autonomous District,Africa/Abidjan,1,https://geocoding-api.open-meteo.com/v1/search...,None,<NA>
2,abidjan__cote_d_ivoire__stade_olympique_alassa...,abidjan__cote_d_ivoire,117860,Stade Olympique Alassane Ouattara,Stade Olympique Alassane Ouattara,Abidjan,Côte d'Ivoire,"Stade Olympique Alassane Ouattara, Abidjan, Cô...",10,2024-01-13,2024-02-11,AFCON 2023,5.35444,-4.00167,found,ok,open-meteo-geocoding,True,False,<NA>,Abidjan,Ivory Coast,CI,Abidjan Autonomous District,Africa/Abidjan,1,https://geocoding-api.open-meteo.com/v1/search...,None,<NA>
3,bouake__cote_d_ivoire__stade_de_bouake,bouake__cote_d_ivoire,1001316,Stade de Bouaké,Stade de Bouaké,Bouake,Côte d'Ivoire,"Stade de Bouaké, Bouake, Côte d'Ivoire",9,2024-01-15,2024-02-07,AFCON 2023,7.69385,-5.03031,found,ok,open-meteo-geocoding,True,False,<NA>,Bouaké,Ivory Coast,CI,Vallée du Bandama District,Africa/Abidjan,4,https://geocoding-api.open-meteo.com/v1/search...,None,<NA>
4,korhogo__cote_d_ivoire__stade_amadou_gon_couli...,korhogo__cote_d_ivoire,1001609,Stade Amadou Gon Coulibaly,Stade Amadou Gon Coulibaly,Korhogo,Côte d'Ivoire,"Stade Amadou Gon Coulibaly, Korhogo, Côte d'Iv...",7,2024-01-16,2024-01-30,AFCON 2023,9.45803,-5.62961,found,ok,open-meteo-geocoding,True,False,<NA>,Korhogo,Ivory Coast,CI,Savanes District,Africa/Abidjan,2,https://geocoding-api.open-meteo.com/v1/search...,None,<NA>
5,san_pedro__cote_d_ivoire__stade_laurent_pokou,san_pedro__cote_d_ivoire,1001497,Stade Laurent Pokou,Stade Laurent Pokou,San Pedro,Côte d'Ivoire,"Stade Laurent Pokou, San Pedro, Côte d'Ivoire",8,2024-01-17,2024-01-30,AFCON 2023,4.74851,-6.63630,found,ok,open-meteo-geocoding,True,False,<NA>,San-Pedro,Ivory Coast,CI,Bas-Sassandra District,Africa/Abidjan,1,https://geocoding-api.open-meteo.com/v1/search...,None,<NA>
6,yamoussoukro__cote_d_ivoire__stade_charles_kon...,yamoussoukro__cote_d_ivoire,1000822,Stade Charles Konan Banny de Yamoussoukro,Stade Charles Konan Banny de Yamoussoukro,Yamoussoukro,Côte d'Ivoire,"Stade Charles Konan Banny de Yamoussoukro, Yam...",8,2024-01-15,2024-02-03,AFCON 2023,6.82055,-5.27674,found,ok,open-meteo-geocoding,True,False,<NA>,Yamoussoukro,Ivory Coast,CI,Lacs District,Africa/Abidjan,2,https://geocoding-api.open-meteo.com/v1/search...,None,<NA>
7,copenhagen__denmark__parken,copenhagen__denmark,495,Parken,Parken,Copenhagen,Denmark,"Parken, Copenhagen, Denmark",4,2021-06-12,2021-06-28,Euro 2020,55.67594,12.56553,found,ok,open-meteo-geocoding,True,False,<NA>,Copenhagen,Denmark,DK,Capital Region,Europe/Copenhagen,2,https://geocoding-api.open-meteo.com/v1/search...,None,<NA>
8,london__england__wembley_stadium,london__england,4666,Wembley Stadium,Wembley Stadium,London,England,"Wembley Stadium, London, England",8,2021-06-13,2021-07-11,Euro 2020,51.50853,-0.12574,found,ok,open-meteo-geocoding,True,False,<NA>,London,United Kingdom,GB,England,Europe/London,1,https://geocoding-api.open-meteo

## Ergebnis BD-10

BD-10 ist erfuellt, wenn die Assertions erfolgreich sind und `data/silver/venue_geocodes.parquet`, `data/cache/geocoding_cache.json` sowie `outputs/tables/geocoding_failures.csv` existieren. Wiederholte Ausfuehrungen nutzen den Cache auf `location_key`-Ebene, damit gleiche Stadt-/Land-Kombinationen nicht mehrfach bei der API angefragt werden.